# encoder
# decoder
# train
# generate

In [1]:
import torch
import torch.nn as nn

In [2]:
chars = "0123456789-/"
special_tokens = ["<SOS>", "<EOS>"]

itos = special_tokens + list(chars)
stoi = {ch: i for i, ch in enumerate(itos)}

SOS_token = stoi["<SOS>"]
EOS_token = stoi["<EOS>"]

vocab_size = len(itos)

print(itos)
print(stoi)

['<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '/']
{'<SOS>': 0, '<EOS>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12, '/': 13}


In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        # 定义了一个RNN网络的输入和输出都是hidden_size
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.1)

        # 你来写 embedding
        # 你来写 rnn

    def forward(self, input):
        # 你来写一次前向传播
        input = self.embedding(input)
        input = self.dropout(input)
        input = input.unsqueeze(1)
        output, hidden = self.rnn(input)
        # hidden.shape = (direction*layer_num,batch_size,diretction*hidden)
        # output.shape = (batch_size,seq_len, last layer of hidden)
        return output, hidden


In [4]:
import random
def make_sample():
    ep = 100
    inputs = []
    targets = []
    for i in range(ep):
        year = random.randint(1950, 2050)
        day = random.randint(1, 28)
        month = random.randint(1, 12)
        input = f"{year}-{month}-{day}"
        inputs.append(input)
        target = f"{day}/{month}/{year}"
        targets.append(target)
    return inputs, targets

inputs, targets = make_sample()
print(inputs[0:10])
print(targets[0:10])


['2002-1-23', '2021-1-12', '1958-6-10', '1980-6-22', '1989-12-7', '2028-11-11', '1961-5-25', '2044-10-18', '2023-10-21', '2014-2-6']
['23/1/2002', '12/1/2021', '10/6/1958', '22/6/1980', '7/12/1989', '11/11/2028', '25/5/1961', '18/10/2044', '21/10/2023', '6/2/2014']


In [13]:
encoder = EncoderRNN(vocab_size, 5)
input_tensor = torch.tensor([stoi[ch] for ch in inputs[0]], dtype=torch.long)
print("input_tensor",input_tensor)
output,hidden = encoder.forward(input_tensor)
# print(output,hidden)

input_tensor tensor([ 4,  2,  2,  4, 12,  3, 12,  4,  5])


In [14]:
print(output.shape)
print(hidden.shape)

torch.Size([9, 1, 5])
torch.Size([1, 9, 5])


In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_output, target_tensor):
        # encoder_output是encoder输出的hiddeng向量
        # target_tensor是decoder输出的向量
        hidden = encoder_output
        batch_size = hidden.shape[1]
        input_token = SOS_token #设置起始词元
        decode_outputs = []
        input_token = torch.full((batch_size, 1), SOS_token, dtype=torch.long)
        for i in range(12):
            hidden, logits, predictIdx = self.forward_step(hidden, input_token)
            decode_outputs.append(logits)

            if target_tensor is not None:
                input_token = target_tensor[:, i].unsqueeze(1)
            if target_tensor is None:
                input_token[0][0] = predictIdx

        decode_hidden = hidden
        return decode_hidden
        
    def forward_step(self, hidden, input_token):
        # input_token.shape是(batch_size,seq_len),元素内容是词元id
        # embedded.shape应当是(batch_size,seq_len,hidden)，元素内容是次元id对应的embedding值
        embedded = self.embedding(input_token)
        # rnn_output 是所有时间步骤的最后一层,shape是 (batch_size,seq_len,hidden)
        # hidden 是最后一个时间步骤的所有层,shape是 (num_layers * num_directions, batch_size, hidden),本例中 num_layers * num_directions 是1
        rnn_output, hidden = self.rnn(embedded, hidden) 
        logits = self.out(hidden[-1]) # out 的输入是最后一维是hidden的任何形状，输出是将输入的最后一维改成和Linear初始化设置的输出一样的形状
        predictIdx = output.argmax(dim = -1, keepdim=True) #argmax沿指定维度找最大值的索引，keepdim=True 使该维度不被消除而是保留为大小1。
        return  hidden, logits, predictIdx

    

In [8]:
hidden_size = 5
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)


In [10]:
print(decoder.out.out_features)

14
